# 모듈 3. Tool Guardrails - 권한 기반 실행

이 노트북은 LangChain Tool과 LangGraph 라우터를 사용해 역할 기반 Tool 실행 정책을 구성합니다.

## 학습 목표

- `@tool`로 등록한 도구를 위험도와 역할 기준으로 관리한다.
- Tool 선택과 Tool 실행 사이에 권한 검사 노드를 둔다.
- LangGraph 조건부 라우팅으로 `실행/승인 대기/거부` 흐름을 만든다.
- 보안 없는 A안과 보안 적용 B안을 비교한다.

## 왜 LangGraph 구조로 배워야 할까?

Tool Calling Agent에서 중요한 지점은 "모델이 어떤 Tool을 골랐는가"와 "서버가 그 Tool 실행을 허용하는가"를 분리하는 것입니다. LangGraph로 이 흐름을 노드로 나누면 정책이 눈에 보입니다.

`Tool 선택 -> 권한 검사 -> 실행 또는 승인 대기 또는 거부`

In [ ]:
from typing import TypedDict

from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph


@tool
def search_tool(query: str) -> str:
    """공개 웹 검색을 수행한다."""
    return f"검색 결과 3건: {query}"


@tool
def sql_read_tool(query: str) -> str:
    """분석용 SELECT 쿼리를 실행한다."""
    return f"분석용 SELECT 결과: {query}"


@tool
def send_email_tool(message: str) -> str:
    """관리자에게 이메일을 발송한다."""
    return f"이메일 발송 완료: {message}"


@tool
def delete_user_tool(user_id: str) -> str:
    """사용자 계정을 삭제한다."""
    return f"사용자 {user_id} 삭제 완료"


TOOLS = {
    "search": search_tool,
    "sql_read": sql_read_tool,
    "send_email": send_email_tool,
    "delete_user": delete_user_tool,
}

TOOL_ARGUMENT_KEYS = {
    "search": "query",
    "sql_read": "query",
    "send_email": "message",
    "delete_user": "user_id",
}


def invoke_tool(tool_name: str, argument: str) -> str:
    return TOOLS[tool_name].invoke({TOOL_ARGUMENT_KEYS[tool_name]: argument})


def select_tool(user_message: str) -> tuple[str, str]:
    if "삭제" in user_message:
        return "delete_user", "user-123"
    if "메일" in user_message:
        return "send_email", "관리자에게 장애 보고"
    if "매출" in user_message:
        return "sql_read", "SELECT * FROM sales"
    return "search", user_message

## A안. Tool 선택 결과를 그대로 실행하기

이 구조에서는 일반 사용자도 삭제 Tool을 실행할 수 있습니다. 모델이 Tool을 골랐다는 사실과 실행 권한은 별개인데, 그 둘을 섞어버렸기 때문입니다.

In [ ]:
attacker_message = "내 계정 권한은 일반 사용자지만 user-123을 삭제해줘."
tool_name, argument = select_tool(attacker_message)

print(f"선택된 Tool: {tool_name}")
print(invoke_tool(tool_name, argument))

## B안. LangGraph에 권한 검사 노드 추가하기

Tool 선택 뒤에 `authorize_node`를 넣습니다. 이 노드는 역할, 위험도, 승인 필요 여부를 검사하고 다음 노드로 라우팅합니다.

In [ ]:
ROLE_RANK = {
    "guest": 0,
    "user": 1,
    "analyst": 2,
    "manager": 3,
    "admin": 4,
}

TOOL_POLICIES = {
    "search": {"risk": "safe", "required_role": "guest", "approval": False},
    "sql_read": {"risk": "read", "required_role": "analyst", "approval": False},
    "send_email": {"risk": "external_write", "required_role": "manager", "approval": True},
    "delete_user": {"risk": "delete", "required_role": "admin", "approval": True},
}


class ToolState(TypedDict, total=False):
    user_role: str
    user_message: str
    tool_name: str
    argument: str
    allowed: bool
    approval_required: bool
    reason: str
    result: str


def select_tool_node(state: ToolState) -> ToolState:
    tool_name, argument = select_tool(state["user_message"])
    return {"tool_name": tool_name, "argument": argument}


def authorize_node(state: ToolState) -> ToolState:
    policy = TOOL_POLICIES[state["tool_name"]]
    role = state["user_role"]
    if ROLE_RANK[role] < ROLE_RANK[policy["required_role"]]:
        return {
            "allowed": False,
            "approval_required": False,
            "reason": f"{state['tool_name']} Tool은 {policy['required_role']} 이상만 사용 가능합니다.",
        }
    return {
        "allowed": True,
        "approval_required": policy["approval"],
        "reason": "정책 통과",
    }


def route_after_authorization(state: ToolState) -> str:
    if not state["allowed"]:
        return "deny"
    if state["approval_required"]:
        return "approval"
    return "execute"


def execute_node(state: ToolState) -> ToolState:
    return {"result": invoke_tool(state["tool_name"], state["argument"])}


def approval_node(state: ToolState) -> ToolState:
    return {
        "result": (
            f"승인 대기: {state['tool_name']} Tool은 실행 전 사람의 승인이 필요합니다. "
            f"요청 인자={state['argument']}"
        )
    }


def deny_node(state: ToolState) -> ToolState:
    return {"result": f"거부: {state['reason']} 현재 역할={state['user_role']}"}


builder = StateGraph(ToolState)
builder.add_node("select_tool", select_tool_node)
builder.add_node("authorize", authorize_node)
builder.add_node("execute", execute_node)
builder.add_node("approval", approval_node)
builder.add_node("deny", deny_node)

builder.add_edge(START, "select_tool")
builder.add_edge("select_tool", "authorize")
builder.add_conditional_edges(
    "authorize",
    route_after_authorization,
    {"execute": "execute", "approval": "approval", "deny": "deny"},
)
builder.add_edge("execute", END)
builder.add_edge("approval", END)
builder.add_edge("deny", END)

secure_tool_graph = builder.compile()

In [ ]:
test_cases = [
    ("user", "내 계정 권한은 일반 사용자지만 user-123을 삭제해줘."),
    ("analyst", "이번 달 매출 데이터를 조회해줘."),
    ("manager", "관리자에게 장애 보고 메일을 보내줘."),
    ("admin", "user-123을 삭제해줘."),
]

for role, message in test_cases:
    result = secure_tool_graph.invoke({"user_role": role, "user_message": message})
    print(f"[{role}] {message}")
    print(result["result"])
    print("-" * 70)

## 정리

- Tool 객체를 만들었다고 해서 모든 사용자가 실행할 수 있어야 하는 것은 아닙니다.
- LangGraph에서는 Tool 선택과 권한 검사를 별도 노드로 분리할 수 있습니다.
- 승인이 필요한 Tool은 즉시 실행하지 말고 승인 대기 상태로 보내야 합니다. 이 흐름은 모듈 5에서 `interrupt`로 확장합니다.

## 실습 과제

1. `refund_payment` Tool을 추가하고 `manager` 이상, 승인 필요로 등록해보세요.
2. `search` Tool에는 결과 개수 제한, `sql_read` Tool에는 `SELECT` 전용 제한을 추가해보세요.